## Лабораторная работа № 6
Самарин НА КИ25-04-3М

Реализуются три операции с использованием Apache Kafka и Apache Spark Structured Streaming:
1. преобразование данных;
2. фильтрация данных;
3. агрегирование данных по временным интервалам.

Для простоты демонстрации каждое сообщение Kafka представляет один заказ с его идентификатором, временем оформления и суммарной стоимостью товарных позиций.

Перед выполнением ноутбука необходимо запустить Kafka через Docker Compose. Инструкция находится в файле `README.md` в архиве проекта.

### 1. Инициализация

In [1]:
import os
import sys

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.3.0 pyspark-shell"
)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LR6_Kafka_Streaming")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.enabled", "false")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.python.worker.faulthandler.enabled", "true")
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

test_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "olist-orders")
    .option("startingOffsets", "earliest")
    .load()
)

test_stream.printSchema()

Spark version: 4.3.0
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [2]:
from pathlib import Path
import json
import time
from datetime import datetime
import sys
import os

from kafka import KafkaProducer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)


DATA_DIR = "data"
OUTPUT_DIR = f"{DATA_DIR}/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORDERS_PATH = f"{DATA_DIR}/orders_clean"
ITEMS_PATH = f"{DATA_DIR}/order_items_clean"

BOOTSTRAP_SERVERS = "localhost:9092"
TOPIC = "olist-orders"

In [3]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"


spark = (
    SparkSession.builder
    .appName("LR6_Kafka_Olist")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.enabled", "false")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.python.worker.faulthandler.enabled", "true")
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = spark.read.parquet(str(ORDERS_PATH))
items = spark.read.parquet(str(ITEMS_PATH))

print("Столбцы заказов:", orders.columns)
print("Столбцы товарных позиций:", items.columns)
print("Количество заказов:", orders.count())
print("Количество товарных позиций:", items.count())

Столбцы заказов: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Столбцы товарных позиций: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
Количество заказов: 99441
Количество товарных позиций: 112650


### 2. Подготовка событий заказов

На этом этапе формируется таблица событий.

In [4]:
required_orders = {"order_id", "order_purchase_timestamp"}
required_items = {"order_id", "price"}

items_by_order = (
    items.groupBy("order_id")
    .agg(F.sum(F.col("price").cast("double")).alias("order_value"))
)

events = (
    orders.select("order_id", "order_purchase_timestamp")
    .join(items_by_order, "order_id", "inner")
    .withColumn("event_time", F.to_timestamp("order_purchase_timestamp"))
    .select(
        F.col("order_id").cast("string"),
        F.date_format("event_time", "yyyy-MM-dd'T'HH:mm:ss").alias("event_time"),
        F.col("order_value").cast("double")
    )
    .filter(F.col("event_time").isNotNull())
)

events.show(10, truncate=False)
print("Количество подготовленных событий:", events.count())

+--------------------------------+-------------------+-----------+
|order_id                        |event_time         |order_value|
+--------------------------------+-------------------+-----------+
|001dbc16dc51075e987543d23a0507c7|2017-01-28T13:17:57|69.9       |
|00ab9fa91aafa3d881164b0da48999aa|2018-05-25T09:47:21|579.0      |
|00b44ba3d7c4a5e9a9ebafef9150781d|2018-08-28T21:10:46|59.9       |
|00b6e66f005a657a6b13eedcacdf9342|2018-08-11T10:47:05|398.0      |
|00b7174ea52bb811ebe2663add51afc0|2018-05-16T09:57:53|69.9       |
|00c9f7d4b0e87781465e562dc109f6aa|2017-05-21T19:21:32|59.96      |
|00e365f4fc03d1098841af23d05c17a6|2018-08-20T09:40:45|175.0      |
|00e4050fe32e786290a29c61141e979f|2017-08-12T12:24:41|49.9       |
|010c995323bc17d8156fbe2c37fed63e|2017-03-25T19:18:00|42.0       |
|012a238ab54294a3b365812ccc82b135|2017-09-07T18:21:44|137.7      |
+--------------------------------+-------------------+-----------+
only showing top 10 rows
Количество подготовленных событий: 98

### 3. Подключение Producer и создание Kafka Topic

Producer отправляет подготовленные события в Kafka в формате JSON. Topic — это именованный поток сообщений.

In [5]:
admin = KafkaAdminClient(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    client_id="lr6-admin"
)

try:
    admin.create_topics([
        NewTopic(name=TOPIC, num_partitions=1, replication_factor=1)
    ])
    print(f"Topic {TOPIC} создан.")
except TopicAlreadyExistsError:
    print(f"Topic {TOPIC} уже существует.")
finally:
    admin.close()

Topic olist-orders уже существует.


### 4. Отправка событий в Kafka

Для демонстрации отправляется ограниченное количество событий. Это имитирует поступление заказов в систему.

In [6]:
LIMIT_EVENTS = 500

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    value_serializer=lambda value: json.dumps(value, ensure_ascii=False).encode("utf-8")
)

rows = (
    events.limit(LIMIT_EVENTS)
    .toLocalIterator()
)

sent = 0
for row in rows:
    message = {
        "order_id": row["order_id"],
        "event_time": row["event_time"],
        "order_value": row["order_value"]
    }
    producer.send(TOPIC, value=message)
    sent += 1

producer.flush()
producer.close()

print(f"В Kafka отправлено сообщений: {sent}")

В Kafka отправлено сообщений: 500


### 5. Чтение потока из Kafka средствами Spark

Spark Structured Streaming подключается к Kafka в режиме потокового чтения.

In [7]:
print("Spark version:", spark.version)

print("Spark JARs:")
print(spark.sparkContext._jsc.sc().listJars().toString())

Spark version: 4.3.0
Spark JARs:
List(spark://127.0.0.1:59568/jars/org.apache.hadoop_hadoop-client-api-3.5.0.jar, spark://127.0.0.1:59568/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.13-4.3.0.jar, spark://127.0.0.1:59568/jars/org.apache.spark_spark-sql-kafka-0-10_2.13-4.3.0.jar, spark://127.0.0.1:59568/jars/com.google.code.findbugs_jsr305-3.0.0.jar, spark://127.0.0.1:59568/jars/at.yawk.lz4_lz4-java-1.11.0.jar, spark://127.0.0.1:59568/jars/org.apache.kafka_kafka-clients-3.9.2.jar, spark://127.0.0.1:59568/jars/org.scala-lang.modules_scala-parallel-collections_2.13-1.2.0.jar, spark://127.0.0.1:59568/jars/org.xerial.snappy_snappy-java-1.1.10.8.jar, spark://127.0.0.1:59568/jars/org.apache.hadoop_hadoop-client-runtime-3.5.0.jar, spark://127.0.0.1:59568/jars/org.apache.commons_commons-pool2-2.13.1.jar, spark://127.0.0.1:59568/jars/org.slf4j_slf4j-api-2.0.17.jar)


In [8]:
print("Версия Spark:", spark.version)

print(
    "Настройка spark.jars.packages:",
    spark.sparkContext.getConf().get("spark.jars.packages", "НЕ ЗАДАНА")
)

print("Загруженные JAR:")
print(spark.sparkContext._jsc.sc().listJars().toString())

Версия Spark: 4.3.0
Настройка spark.jars.packages: НЕ ЗАДАНА
Загруженные JAR:
List(spark://127.0.0.1:59568/jars/org.apache.hadoop_hadoop-client-api-3.5.0.jar, spark://127.0.0.1:59568/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.13-4.3.0.jar, spark://127.0.0.1:59568/jars/org.apache.spark_spark-sql-kafka-0-10_2.13-4.3.0.jar, spark://127.0.0.1:59568/jars/com.google.code.findbugs_jsr305-3.0.0.jar, spark://127.0.0.1:59568/jars/at.yawk.lz4_lz4-java-1.11.0.jar, spark://127.0.0.1:59568/jars/org.apache.kafka_kafka-clients-3.9.2.jar, spark://127.0.0.1:59568/jars/org.scala-lang.modules_scala-parallel-collections_2.13-1.2.0.jar, spark://127.0.0.1:59568/jars/org.xerial.snappy_snappy-java-1.1.10.8.jar, spark://127.0.0.1:59568/jars/org.apache.hadoop_hadoop-client-runtime-3.5.0.jar, spark://127.0.0.1:59568/jars/org.apache.commons_commons-pool2-2.13.1.jar, spark://127.0.0.1:59568/jars/org.slf4j_slf4j-api-2.0.17.jar)


In [9]:
spark.stop()

# os.environ["PYSPARK_SUBMIT_ARGS"] = (
#     "--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.3.0 pyspark-shell"
# )

#     # .config(
#     #     "spark.jars.packages",
#     #     "org.apache.spark:spark-sql-kafka-0-10_2.13:4.3.0"
#     # )

# spark = (
#     SparkSession.builder
#     .appName("LR6_Kafka_Streaming")
#     .master("local[*]")
#     .config("spark.driver.host", "127.0.0.1")
#     .config("spark.driver.bindAddress", "127.0.0.1")
#     .config("spark.sql.shuffle.partitions", "8")
#     .config("spark.ui.enabled", "false")
#     .config("spark.pyspark.python", sys.executable)
#     .config("spark.pyspark.driver.python", sys.executable)
#     .config("spark.python.worker.faulthandler.enabled", "true")
#     .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
#     .getOrCreate()
# )

import os
import sys

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.3.0 pyspark-shell"
)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LR6_Kafka_Streaming")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.enabled", "false")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.python.worker.faulthandler.enabled", "true")
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("order_value", DoubleType(), True)
])

kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "olist-orders")
    .option("startingOffsets", "earliest")
    .load()
)

parsed_stream = (
    kafka_stream
    .select(F.from_json(F.col("value").cast("string"), schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", F.to_timestamp("event_time"))
)

print("Поток Kafka подготовлен к обработке.")

Поток Kafka подготовлен к обработке.


### 6. Преобразование и фильтрация данных

Преобразование включает приведение типов и создание дополнительного поля `order_date`. Фильтрация оставляет только корректные события: с заполненным идентификатором, временем и положительной стоимостью заказа.

In [10]:
transformed_stream = (
    parsed_stream
    .withColumn("order_value", F.col("order_value").cast("double"))
    .withColumn("order_date", F.to_date("event_time"))
    .withColumn("order_hour", F.hour("event_time"))
)

filtered_stream = transformed_stream.filter(
    F.col("order_id").isNotNull()
    & F.col("event_time").isNotNull()
    & F.col("order_value").isNotNull()
    & (F.col("order_value") > 0)
)

print("Операции преобразования и фильтрации описаны.")

Операции преобразования и фильтрации описаны.


### 7. Агрегирование потока

Для демонстрации рассчитываются количество заказов и суммарная стоимость заказов в минутных временных интервалах. В реальной системе такие показатели могут использоваться для мониторинга активности интернет-магазина.

In [11]:
aggregated_stream = (
    filtered_stream
    .withWatermark("event_time", "10 minutes")
    .groupBy(F.window("event_time", "1 minute"))
    .agg(
        F.count("order_id").alias("orders_count"),
        F.round(F.sum("order_value"), 2).alias("total_revenue"),
        F.round(F.avg("order_value"), 2).alias("average_order_value")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "orders_count",
        "total_revenue",
        "average_order_value"
    )
)

print("Поток агрегированных показателей сформирован.")

Поток агрегированных показателей сформирован.


### 8. Запуск потоковой обработки и сохранение результата

Результат агрегирования записывается в консоль и в Parquet.

Так как исходный Producer отправляет ограниченное число сообщений, обработка запускается на ограниченное время. Если сообщений ещё нет, можно повторно выполнить ячейку Producer, пока потоковая задача работает.

In [12]:
CHECKPOINT_DIR = f"{OUTPUT_DIR}/kafka_checkpoint"
RESULT_DIR = f"{OUTPUT_DIR}/kafka_aggregates"

query = (
    aggregated_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", RESULT_DIR)
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Потоковая обработка запущена. Ожидание до 60 секунд...")
query.awaitTermination(60)
query.stop()
print("Потоковая обработка остановлена.")

Потоковая обработка запущена. Ожидание до 60 секунд...
Потоковая обработка остановлена.


### 9. Просмотр результатов

После завершения потоковой обработки читаем сохранённые Parquet-файлы и выводим агрегированные показатели.

In [13]:
result_path = Path(RESULT_DIR)

if result_path.exists() and list(result_path.glob("*.parquet")):
    result = spark.read.parquet(RESULT_DIR)
    result.orderBy("window_start").show(50, truncate=False)
    print("Количество агрегированных интервалов:", result.count())
else:
    print("Parquet-результаты не найдены.")

+-------------------+-------------------+------------+-------------+-------------------+
|window_start       |window_end         |orders_count|total_revenue|average_order_value|
+-------------------+-------------------+------------+-------------+-------------------+
|2016-10-04 14:44:00|2016-10-04 14:45:00|7           |1749.3       |249.9              |
|2016-10-07 19:02:00|2016-10-07 19:03:00|7           |1743.0       |249.0              |
|2016-10-09 12:43:00|2016-10-09 12:44:00|7           |615.93       |87.99              |
|2017-01-15 00:01:00|2017-01-15 00:02:00|7           |146.86       |20.98              |
|2017-01-18 14:32:00|2017-01-18 14:33:00|7           |495.81       |70.83              |
|2017-01-20 22:18:00|2017-01-20 22:19:00|7           |279.3        |39.9               |
|2017-01-22 23:07:00|2017-01-22 23:08:00|7           |167.3        |23.9               |
|2017-01-28 13:17:00|2017-01-28 13:18:00|7           |489.3        |69.9               |
|2017-02-03 19:23:00|

In [14]:
spark.stop()
print("Spark остановлен.")

Spark остановлен.


### 10. Вывод

На основе очищенных данных Olist были сформированы события заказов и отправлены в Kafka в формате JSON. Spark Structured Streaming считал сообщения из Kafka и выполнил три основные операции: преобразование полей и типов данных, фильтрацию некорректных событий, агрегирование количества и стоимости заказов по временным интервалам.

Результаты потоковой обработки сохранены в формате Parquet. Таким образом, продемонстрирован базовый конвейер потоковой обработки данных.